In [70]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

import math

from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score

from pathlib import Path
import time

In [2]:
train_df = pd.read_csv("../data/train_split.csv")
val_df = pd.read_csv("../data/val_split.csv")

In [3]:
print("Training Shape:", train_df.shape)
print("Validation Shape:", val_df.shape)

print("\nTraining Columns:", train_df.columns.tolist())
print("Validation Columns:", val_df.columns.tolist())

display(train_df.head())


Training Shape: (3200, 2)
Validation Shape: (800, 2)

Training Columns: ['text', 'label']
Validation Columns: ['text', 'label']


,text,label
0,new fcc rule bring highspeed internet american...,3
1,gazprom decides bid yukos key oil unit moscow ...,2
2,treasury upbeat fed minute chicago reuters u t...,2
3,man find troop body glacier ap ap amateur ital...,0
4,basketball seattle storm win wnba final seattl...,1


In [4]:
print("Training Missing Values:")
print(train_df[["text", "label"]].isnull().sum())

print("\nValidation Missing Values:")
print(val_df[["text", "label"]].isnull().sum())

print("\nTraining Class Distribution:")
print(train_df["label"].value_counts().sort_index())

print("\nValidation Class Distribution:")
print(val_df["label"].value_counts().sort_index())

Training Missing Values:
text     0
label    0
dtype: int64

Validation Missing Values:
text     0
label    0
dtype: int64

Training Class Distribution:
label
0    800
1    800
2    800
3    800
Name: count, dtype: int64

Validation Class Distribution:
label
0    200
1    200
2    200
3    200
Name: count, dtype: int64


In [6]:
train_empty = train_df["text"].str.strip().eq("").sum()
val_empty = val_df["text"].str.strip().eq("").sum()

print("Training Empty Texts:", train_empty)
print("Validation Empty Texts:", val_empty)


train_texts = set(train_df["text"].str.strip().str.lower())
val_texts = set(val_df["text"].str.strip().str.lower())

common_texts = train_texts.intersection(val_texts)

print("Common News Count:", len(common_texts))

Training Empty Texts: 0
Validation Empty Texts: 0
Common News Count: 0


In [9]:
def tokenize(text):
    return text.lower().split()


In [10]:
PAD_ID = 0
UNK_ID = 1

def build_vocab(texts):
    vocab = {
        "[PAD]": PAD_ID,
        "[UNK]": UNK_ID
    }

    for text in texts:
        for token in tokenize(text):
            if token not in vocab:
                vocab[token] = len(vocab)

    return vocab

In [14]:
vocab = build_vocab(train_df["text"])

print("Vocabulary Size:", len(vocab))
print("First 10 Tokens:", list(vocab.items())[:10])

Vocabulary Size: 7787
First 10 Tokens: [('[PAD]', 0), ('[UNK]', 1), ('new', 2), ('fcc', 3), ('rule', 4), ('bring', 5), ('highspeed', 6), ('internet', 7), ('american', 8), ('jennifer', 9)]


In [22]:
MAX_LENGTH = 64

def encode_text(text, vocab, max_length):
    tokens = tokenize(text)

    token_ids = []

    for token in tokens:
        token_ids.append(vocab.get(token , UNK_ID))

    token_ids = token_ids[:max_length]

    padding_length = max_length - len(token_ids)

    token_ids = token_ids + [PAD_ID] * padding_length


    return token_ids

In [26]:
train_encoded = [
    encode_text(text, vocab, MAX_LENGTH)
    for text in train_df["text"]
]

val_encoded = [
    encode_text(text, vocab, MAX_LENGTH)
    for text in val_df["text"]
]

In [28]:
print("Haber sayısı:", len(train_encoded))
print("İlk haberin uzunluğu:", len(train_encoded[0]))


print("Haber sayısı:", len(val_encoded))
print("İlk haberin uzunluğu:", len(val_encoded[0]))

Haber sayısı: 3200
İlk haberin uzunluğu: 64
Haber sayısı: 800
İlk haberin uzunluğu: 64


In [ ]:
X_train = torch.tensor(train_encoded ,dtype=torch.long)
y_train = torch.tensor(train_df["label"] ,dtype=torch.long)

X_val = torch.tensor(val_encoded ,dtype=torch.long)
y_val = torch.tensor(val_df["label"] ,dtype=torch.long)

In [40]:
print("X_train:", X_train.shape, X_train.dtype)
print("X_val:", X_val.shape, X_val.dtype)


print("X_train:", y_train.shape, y_train.dtype)
print("X_val:", y_val.shape, y_val.dtype)

X_train: torch.Size([3200, 64]) torch.int64
X_val: torch.Size([800, 64]) torch.int64
X_train: torch.Size([3200]) torch.int64
X_val: torch.Size([800]) torch.int64


In [43]:
print(torch.unique(y_train))
print(torch.unique(y_val))

tensor([0, 1, 2, 3])
tensor([0, 1, 2, 3])


In [48]:
def positional_encoding(seq_len, d_model):

    positions = torch.arange(seq_len).unsqueeze(1)

    div_term = torch.exp(
        torch.arange(0, d_model, 2).float()
        * (-math.log(10000.0) / d_model)
    )
    angles = positions * div_term

    PE = torch.zeros(seq_len , d_model)
    PE[:, 0::2] = torch.sin(angles)
    PE[:, 1::2] = torch.cos(angles)

    return PE

In [ ]:
class NewsRouteTransformer(nn.Module):

    def __init__(
        self,
        vocab_size,
        d_model=64,
        num_heads=4,
        num_layers=1,
        num_classes=4
    ):
        super().__init__()

        # 1. Embedding
        self.embedding = nn.Embedding(
            vocab_size,
            d_model,
            padding_idx=PAD_ID
        )

        # 2. Positional Encoding
        self.register_buffer(
            "pe",
            positional_encoding(MAX_LENGTH, d_model)
        )

        # 3. Transformer Encoder Layer
        self.encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=128,
            dropout=0.1,
            batch_first=True
        )

        # 4. Transformer Encoder
        self.encoder = nn.TransformerEncoder(
            encoder_layer=self.encoder_layer,
            num_layers=num_layers
        )

        # 5. Classification Head
        self.classifier = nn.Linear(
            in_features=d_model,
            out_features=num_classes
        )

    def forward(self, input_ids):

        # 1. Padding Mask
        padding_mask = input_ids == PAD_ID

        # 2. Embedding
        x = self.embedding(input_ids)

        # 3. Positional Encoding
        x = x + self.pe[:input_ids.size(1)].unsqueeze(0)

        # 4. Transformer Encoder
        x = self.encoder(
            src=x,
            src_key_padding_mask=padding_mask
        )

        # 5. Masked Mean Pooling
        valid_mask = input_ids != PAD_ID

        mask = valid_mask.unsqueeze(-1).float()

        mask_x = x * mask

        token_sum = mask_x.sum(dim=1)

        token_count = mask.sum(dim=1).clamp(min=1)

        pooled = token_sum / token_count

        # 6. Classification Head
        logits = self.classifier(pooled)

        return logits

In [ ]:
torch.manual_seed(42)

model = NewsRouteTransformer(vocab_size=len(vocab))
model.eval()

input_ids = X_train[:32]


with torch.no_grad():

    logits = model(input_ids)

    embedding = model.embedding(input_ids)

    padding_mask = input_ids == PAD_ID

    encoder_output = model.encoder(
        src=embedding + model.pe[:input_ids.size(1)].unsqueeze(0),
        src_key_padding_mask=padding_mask
    )

    mask = (~padding_mask).unsqueeze(-1).float()

    pooled = (
        (encoder_output * mask).sum(dim=1)
        / mask.sum(dim=1).clamp(min=1)
    )

# 4. Shape kontrolleri
print("Input IDs Shape:", input_ids.shape)
print("Embedding Shape:", embedding.shape)
print("Encoder Output Shape:", encoder_output.shape)
print("Pooled Output Shape:", pooled.shape)
print("Logits Shape:", logits.shape)

# 5. NaN / Inf kontrolleri
print("\nNaN:", torch.isnan(logits).any().item())
print("Inf:", torch.isinf(logits).any().item())
print("Finite:", torch.isfinite(logits).all().item())

Input IDs Shape: torch.Size([32, 64])
Embedding Shape: torch.Size([32, 64, 64])
Encoder Output Shape: torch.Size([32, 64, 64])
Pooled Output Shape: torch.Size([32, 64])
Logits Shape: torch.Size([32, 4])

NaN: False
Inf: False
Finite: True


/home/pasali/anaconda3/envs/myenv/lib/python3.14/site-packages/torch/nn/modules/transformer.py:533: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/NestedTensorImpl.cpp:177.)
  output = torch._nested_tensor_from_mask(


In [60]:
BATCH_SIZE = 32

train_dataset = TensorDataset(X_train, y_train)


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

In [61]:
LEARNING_RATE = 0.001
EPOCHS = 3

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE
)

In [62]:
print("Batch sayısı:", len(train_loader))
print("Epoch sayısı:", EPOCHS)

Batch sayısı: 100
Epoch sayısı: 3


In [63]:

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0.0

    for input_ids, labels in train_loader:

        # 1. Gradient sıfırla
        optimizer.zero_grad()

        # 2. Forward Pass
        logits = model(input_ids)
        # 3. Loss hesapla
        loss = criterion(logits , labels)

        # 4. Backpropagation
        loss.backward()

        # 5. Optimizer step
        optimizer.step()
        # 6. Loss değerini kaydet
        total_loss += loss.item()
        
        

    # Epoch ortalama loss değerini hesapla.
    avg_loss = total_loss / len(train_loader)

    print(f"Epoch {epoch + 1} / {EPOCHS}")
    print(f"Train Loss: {avg_loss:.4f}\n")
    

Epoch 1 / 3
Train Loss: 1.2868

Epoch 2 / 3
Train Loss: 0.8547

Epoch 3 / 3
Train Loss: 0.5855



In [64]:
val_dataset = TensorDataset(X_val, y_val)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

In [66]:
model.eval()

all_preds = []
all_labels = []
total_val_loss = 0.0

with torch.no_grad():

    for input_ids, labels in val_loader:

        logits = model(input_ids)

     
        loss = criterion(logits, labels)

        total_val_loss += loss.item() * labels.size(0)

        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())


validation_loss = total_val_loss / len(val_dataset)

validation_accuracy = accuracy_score(
    all_labels, all_preds
)

validation_macro_f1 = f1_score(
    all_labels,
    all_preds,
    average="macro"
)

print(f"Validation Accuracy: {validation_accuracy:.4f}")
print(f"Validation Macro-F1: {validation_macro_f1:.4f}")
print(f"Validation Loss: {validation_loss:.4f}")

Validation Accuracy: 0.7137
Validation Macro-F1: 0.7108
Validation Loss: 0.7518


In [ ]:
reports_dir = Path("../reports")
reports_dir.mkdir(parents=True, exist_ok=True)


training_metrics = pd.DataFrame({
    "epoch": [1, 2, 3],
    "train_loss": [1.2868, 0.8547, 0.5855],
    "epoch_seconds": [None, None, None]
})

training_metrics.to_csv(
    reports_dir / "transformer_training_metrics.csv",
    index=False
)

# Validation Metrics
validation_metrics = pd.DataFrame([{
    "model": "NewsRouteTransformer",
    "train_samples": len(X_train),
    "val_samples": len(X_val),
    "accuracy": validation_accuracy,
    "macro_f1": validation_macro_f1,
    "validation_loss": validation_loss
}])

validation_metrics.to_csv(
    reports_dir / "transformer_validation.csv",
    index=False
)

print("Training metrics kaydedildi.")
print("Validation metrics kaydedildi.")

Training metrics kaydedildi.
Validation metrics kaydedildi.


In [69]:
models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

model_path = models_dir / "newsroute_transformer.pt"

torch.save(
    model.state_dict(),
    model_path
)

print(f"Model ağırlıkları kaydedildi: {model_path}")

Model ağırlıkları kaydedildi: ../models/newsroute_transformer.pt


In [71]:
device = next(model.parameters()).device


torch.manual_seed(42)

timing_model = NewsRouteTransformer(
    vocab_size=len(vocab),
    d_model=64,
    num_heads=4,
    num_layers=1,
    num_classes=4
).to(device)

timing_optimizer = torch.optim.AdamW(
    timing_model.parameters(),
    lr=LEARNING_RATE
)

timing_results = []

if device.type == "cuda":
    torch.cuda.synchronize()

total_start = time.perf_counter()

for epoch in range(EPOCHS):

    timing_model.train()
    total_loss = 0.0

    if device.type == "cuda":
        torch.cuda.synchronize()

    start_time = time.perf_counter()

    for input_ids, labels in train_loader:

        input_ids = input_ids.to(device)
        labels = labels.to(device)

        timing_optimizer.zero_grad()

        logits = timing_model(input_ids)

        loss = criterion(logits, labels)

        loss.backward()

        timing_optimizer.step()

        total_loss += loss.item()

    if device.type == "cuda":
        torch.cuda.synchronize()

    epoch_seconds = time.perf_counter() - start_time

    avg_loss = total_loss / len(train_loader)

    timing_results.append({
        "epoch": epoch + 1,
        "train_loss": avg_loss,
        "epoch_seconds": epoch_seconds
    })

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"Loss: {avg_loss:.4f} | "
        f"Time: {epoch_seconds:.2f}s"
    )

total_seconds = time.perf_counter() - total_start

print(f"\nDonanım: {device}")
print(f"Toplam eğitim süresi: {total_seconds:.2f}s")

Epoch 1/3 | Loss: 1.2868 | Time: 1.05s
Epoch 2/3 | Loss: 0.8547 | Time: 1.00s
Epoch 3/3 | Loss: 0.5855 | Time: 1.00s

Donanım: cpu
Toplam eğitim süresi: 3.05s


In [ ]:
timing_df = pd.DataFrame(timing_results)

print(timing_df.round(4))

print(f"\nToplam süre: {total_seconds:.2f}s")

timing_df.to_csv(
    "../reports/transformer_timing_check.csv",
    index=False
)

   epoch  train_loss  epoch_seconds
0      1      1.2868         1.0514
1      2      0.8547         1.0007
2      3      0.5855         0.9952

Toplam süre: 3.05s


In [74]:
batch_input_ids, batch_labels = next(iter(train_loader))

padding_mask = batch_input_ids == PAD_ID


padding_count = padding_mask.sum().item()
valid_count = (~padding_mask).sum().item()

# Kontroller
print("Batch Shape:", batch_input_ids.shape)
print("Padding Token Count:", padding_count)
print("Valid Token Count:", valid_count)
print("Padding Mask Shape:", padding_mask.shape)

print("\n--- Kontroller ---")

print("PAD ID:", PAD_ID)

print(
    "Boolean Tensor:",
    padding_mask.dtype == torch.bool
)

print(
    "Mask Shape Doğru:",
    padding_mask.shape == batch_input_ids.shape
)

print(
    "PAD konumları True:",
    torch.equal(padding_mask, batch_input_ids == 0)
)

# Masked Pooling kontrolü
valid_mask = ~padding_mask

mask = valid_mask.unsqueeze(-1).float()

print(
    "Pooling PAD konumlarını dışlıyor:",
    bool((mask.squeeze(-1)[padding_mask] == 0).all())
)

Batch Shape: torch.Size([32, 64])
Padding Token Count: 1272
Valid Token Count: 776
Padding Mask Shape: torch.Size([32, 64])

--- Kontroller ---
PAD ID: 0
Boolean Tensor: True
Mask Shape Doğru: True
PAD konumları True: True
Pooling PAD konumlarını dışlıyor: True


In [75]:
# GÖREV 8 — Üç Otomatik Test

model.eval()

batch_input_ids, batch_labels = next(iter(train_loader))

# TEST 1 — Forward Shape
with torch.no_grad():
    logits = model(batch_input_ids)

assert logits.shape == (batch_input_ids.size(0), 4)

print("Test 1 — Forward Shape: PASSED")


# TEST 2 — Padding Mask
padding_mask = batch_input_ids == PAD_ID

assert padding_mask.dtype == torch.bool
assert padding_mask.shape == batch_input_ids.shape
assert torch.equal(padding_mask, batch_input_ids == 0)

print("Test 2 — Padding Mask: PASSED")


# TEST 3 — Finite Loss
with torch.no_grad():
    logits = model(batch_input_ids)
    loss = criterion(logits, batch_labels)

assert torch.isfinite(loss).item()

print("Test 3 — Finite Loss: PASSED")

Test 1 — Forward Shape: PASSED
Test 2 — Padding Mask: PASSED
Test 3 — Finite Loss: PASSED
